In [1]:
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import eigs
from scipy.stats import linregress
import mpmath
from numba import njit
import time

# ======== 1. 终极参数 ========
U_C = 1.543689012  
K_OPT = 12.32      
STEPS = 10**8      # 跑 1 亿步 (兼顾速度与精度)
N_BINS = 10000     # 10000 分辨率
N_ZEROS = 100      # 我们就要看最难的前 100 个！

# ======== 2. 召唤大自然的真实底牌 ========
mpmath.mp.dps = 15
true_zeros = np.array([float(mpmath.zetazero(i).imag) for i in range(1, N_ZEROS + 1)])

# ======== 3. Numba 极速演化 ========
@njit
def run_universe(steps, n_bins, u_c, k_opt):
    transitions = np.zeros((n_bins, n_bins), dtype=np.uint32)
    x = 0.5
    last_bin = int((x + 1.0) / 2.0 * (n_bins - 1))
    
    for i in range(1, steps + 1):
        mu = u_c + k_opt / (np.log(i + 100)**2)
        x = 1.0 - mu * x * x
        if x > 1.0: x = 0.999
        elif x < -1.0: x = -0.999
        current_bin = int((x + 1.0) / 2.0 * (n_bins - 1))
        transitions[last_bin, current_bin] += 1
        last_bin = current_bin
    return transitions

print(f"🚀 正在运行 1 亿步非自治演化以提取前 {N_ZEROS} 个零点...")
start_time = time.time()
trans_matrix = run_universe(STEPS, N_BINS, U_C, K_OPT)

# ======== 4. 提取与解卷绕 ========
P_sparse = sp.csr_matrix(trans_matrix, dtype=np.float64)
row_sums = np.array(P_sparse.sum(axis=1)).flatten()
row_sums[row_sums == 0] = 1.0 
P_sparse.data /= row_sums[P_sparse.indices]

# 抓取特征值
eigenvalues, _ = eigs(P_sparse, k=N_ZEROS + 50, which='LM', tol=1e-5)
phases = np.angle(eigenvalues)
valid_indices = (np.abs(eigenvalues.imag) > 1e-4) 
sorted_phases = np.sort(phases[valid_indices])
unwrapped_phases = np.unwrap(sorted_phases)[:N_ZEROS]

# ======== 5. 宏观对齐与预测 ========
# 我们用整体趋势找到物理空间到黎曼空间的映射比例
slope, intercept, _, _, _ = linregress(unwrapped_phases, true_zeros)
predicted_zeros = slope * unwrapped_phases + intercept

# ======== 6. 极度硬核的对撞报表 ========
print(f"✅ 演化完成！耗时: {time.time() - start_time:.2f} 秒\n")
print("="*65)
print(f"   【创世计算器：前 100 个黎曼零点预测对撞报表】")
print("="*65)
print(f"{'Index (n)':<10} | {'预测位置 (Predicted)':<20} | {'真实位置 (True)':<15} | {'绝对误差 (Error)'}")
print("-" * 65)

# 打印前 20 个和最后几个，中间省略以保持清爽
for i in range(20):
    err = abs(predicted_zeros[i] - true_zeros[i])
    print(f"n = {i+1:<5} | {predicted_zeros[i]:<20.4f} | {true_zeros[i]:<15.4f} | {err:.4f}")

print("... (中间省略) ...")

for i in range(N_ZEROS - 5, N_ZEROS):
    err = abs(predicted_zeros[i] - true_zeros[i])
    print(f"n = {i+1:<5} | {predicted_zeros[i]:<20.4f} | {true_zeros[i]:<15.4f} | {err:.4f}")
print("="*65)
mean_err = np.mean(np.abs(predicted_zeros - true_zeros))
print(f"🎯 前 {N_ZEROS} 个零点的平均绝对误差: {mean_err:.4f}")

🚀 正在运行 1 亿步非自治演化以提取前 100 个零点...
✅ 演化完成！耗时: 5.28 秒

   【创世计算器：前 100 个黎曼零点预测对撞报表】
Index (n)  | 预测位置 (Predicted)     | 真实位置 (True)     | 绝对误差 (Error)
-----------------------------------------------------------------
n = 1     | 31.9232              | 14.1347         | 17.7885
n = 2     | 36.8226              | 21.0220         | 15.8006
n = 3     | 39.4127              | 25.0109         | 14.4019
n = 4     | 40.6744              | 30.4249         | 10.2495
n = 5     | 44.6657              | 32.9351         | 11.7306
n = 6     | 46.0465              | 37.5862         | 8.4603
n = 7     | 47.4422              | 40.9187         | 6.5235
n = 8     | 49.4604              | 43.3271         | 6.1334
n = 9     | 51.7831              | 48.0052         | 3.7780
n = 10    | 51.8949              | 49.7738         | 2.1210
n = 11    | 55.3462              | 52.9703         | 2.3759
n = 12    | 57.7704              | 56.4462         | 1.3241
n = 13    | 60.6814              | 59.3470         | 1.3343
n 